# Online scoring

Step 8 checks the production path: transactions are scored **one at a time**, using each customer's history from the SQLite database.

| Section | Question |
|---|---|
| Step 8a | Is the history database complete, minimal and fast to query? |
| Step 8b | Does scoring one transaction at a time give exactly the same features and scores as batch training? |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option("display.float_format", "{:,.4f}".format)

from sentinel.data_sources import MODEL_DIR, load_raw
from sentinel.features.pipeline import FEATURE_COLUMNS, build_features
from sentinel.model.artifacts import load_model
from sentinel.model.split import split_by_time

df = load_raw()
features = build_features(df)
test = split_by_time(features)["test"]
booster, meta = load_model(MODEL_DIR)

print("test rows:", len(test))
print("model thresholds:", round(meta["low_threshold"], 4), round(meta["high_threshold"], 4))

test rows: 229040
model thresholds: 0.0234 0.7185


## Step 8a: the history database

`scripts/build_history_db.py` seeds `data/history.db` (SQLite) with every transaction **before 1 July 2020**, the start of the replay period.

- **Data minimisation:** only the 12 columns needed for scoring and investigation are stored; names, gender, date of birth and job are left out.
- **Speed:** an index on `(cc_num, trans_date_trans_time)` makes "this customer's last 30 days" a direct lookup.

**Checks:** row count and date range, one customer's latest purchases, and that queries use the index.

In [2]:
import sqlite3

from sentinel.data_sources import HISTORY_DB_PATH

conn = sqlite3.connect(f"file:{HISTORY_DB_PATH}?mode=ro", uri=True)

print(pd.read_sql_query(
    "SELECT COUNT(*) AS rows, MIN(trans_date_trans_time) AS first, "
    "MAX(trans_date_trans_time) AS last FROM transactions", conn))
print()
print(pd.read_sql_query(
    "SELECT trans_date_trans_time, amt, category FROM transactions "
    "WHERE cc_num = ? ORDER BY trans_date_trans_time DESC LIMIT 5",
    conn, params=(2703186189652095,)))
print()
print(pd.read_sql_query(
    "EXPLAIN QUERY PLAN SELECT * FROM transactions "
    "WHERE cc_num = ? AND trans_date_trans_time >= ?",
    conn, params=(2703186189652095, "2020-06-01"))["detail"].iloc[0])

conn.close()

      rows                first                 last
0  1326733  2019-01-01 00:00:18  2020-06-30 23:59:50

  trans_date_trans_time      amt       category
0   2020-06-30 19:22:37  51.6200      kids_pets
1   2020-06-30 19:06:22  41.7500           home
2   2020-06-30 13:45:33 340.9500   shopping_net
3   2020-06-30 13:17:38  35.9300  personal_care
4   2020-06-30 08:10:40 207.6500    grocery_pos

SEARCH transactions USING INDEX idx_customer_time (cc_num=? AND trans_date_trans_time>?)


## Step 8b: online features = batch features

For 500 random test transactions, features are computed two ways:

| Way | How | Used in |
|---|---|---|
| **Batch** | `build_features()` on the full dataset at once | Training |
| **Online** | `score_transaction()`: the customer's last 30 days from SQLite, plus the new transaction | Production (replay, API) |

Any difference would be **training–serving skew**. The database is opened read-only.

**Checks:** identical features, missing values in the same places, identical categories and identical scores.

In [3]:
from sentinel.scoring import score_transaction

sample = test.sample(500, random_state=42)

conn = sqlite3.connect(f"file:{HISTORY_DB_PATH}?mode=ro", uri=True)
online_features, online_scores = [], []
for _, row in sample.iterrows():
    result = score_transaction(conn, row[df.columns].to_dict(), booster, meta)
    online_features.append(result["features"])
    online_scores.append(result["score"])
conn.close()

online = pd.DataFrame(online_features, index=sample.index)
batch = sample[FEATURE_COLUMNS]
numeric = [c for c in FEATURE_COLUMNS if c != "category"]

feature_diff = (online[numeric].astype(float) - batch[numeric].astype(float)).abs().max().max()
nan_match = (online[numeric].isna() == batch[numeric].isna()).all().all()
category_match = (online["category"].astype(str) == batch["category"].astype(str)).all()
score_diff = np.abs(np.array(online_scores) - booster.predict(batch)).max()

print("largest feature difference:", feature_diff)
print("missing values in the same places:", nan_match)
print("categories identical:", category_match)
print("largest score difference:", score_diff)

largest feature difference: 2.842170943040401e-14
missing values in the same places: True
categories identical: True
largest score difference: 0.0
